# Huấn luyện LoRA với Unsloth trên dữ liệu VihealthQA
Notebook này sử dụng cấu hình giống với file `train_qwen_lora_unsloth (1).ipynb` (ViMMRC) để tối ưu hóa quá trình fine-tuning cho model Qwen2.5-1.5B-Instruct với Chat Template chuẩn.

In [9]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

Found existing installation: torch 2.12.1+cu132
Uninstalling torch-2.12.1+cu132:
  Successfully uninstalled torch-2.12.1+cu132
Found existing installation: torchvision 0.27.1+cu132
Uninstalling torchvision-0.27.1+cu132:
  Successfully uninstalled torchvision-0.27.1+cu132
Found existing installation: xformers 0.0.35
Uninstalling xformers-0.0.35:
  Successfully uninstalled xformers-0.0.35
Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: trl 0.24.0
Uninstalling trl-0.24.0:
  Successfully uninstalled trl-0.24.0
Found existing installation: unsloth 2026.6.9
Uninstalling unsloth-2026.6.9:
  Successfully uninstalled unsloth-2026.6.9
Found existing installation: unsloth_zoo 2026.6.7
Uninstalling unsloth_zoo-2026.6.7:
  Successfully uninstalled unsloth_zoo-2026.6.7


In [10]:
# Cài đặt lại bản PyTorch ổn định
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 114.6 MB/s  0:00:0500:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 116.8 MB/s  0:00:0200:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 106.8 MB/s  0:00:02a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 117.6 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 115.6 MB/s  0:00:0500:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 156.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 103.0 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 331.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 118.1 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 115.9 MB/s

In [11]:
# Cài đặt các thư viện vệ tinh
!pip install transformers trl peft accelerate bitsandbytes xformers --no-cache-dir

# Cài đặt Unsloth mới nhất từ nhánh chính
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 98.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 129.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 344.2 MB/s  0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [trl]3/4 [trl]sformers]

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-4xbi85fw/unsloth_ae7e68252dd84594babcbec016fa4dd1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-4xbi85fw/unsloth_ae7e68252dd84594babcbec016fa4dd1
  Resolved https://github.com/unslothai/unsloth.git to commit 91

In [1]:
import torch

print("--- KIỂM TRA PHIÊN BẢN CÁC THƯ VIỆN ---")
print(f"PyTorch version:     {torch.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")

--- KIỂM TRA PHIÊN BẢN CÁC THƯ VIỆN ---
PyTorch version:     2.11.0+cu128
CUDA Available:      True


In [2]:
# 1. Khởi tạo Mô hình và Tokenizer
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Thiết lập cấu hình LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.476 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [3]:
# 2. Tiền xử lý dữ liệu VihealthQA
import pandas as pd
from datasets import Dataset

train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('val.csv')

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [4]:
# 3. Format prompt theo định dạng chuẩn của ChatML
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    questions = examples["question"]
    answers   = examples["answer"]
    texts = []
    for question, answer in zip(questions, answers):
        messages = [
            {"role": "system", "content": "Bạn là một chuyên gia y tế AI thông minh, tận tâm và có khả năng giải đáp các vấn đề sức khỏe một cách chính xác."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": str(answer)}
        ]
        # tokenizer.apply_chat_template sẽ tự thêm EOS token
        text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
        texts.append(text)
    return { "text" : texts }

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
val_dataset = val_dataset.map(formatting_prompts_func, batched = True)

In [5]:
# 4. Cấu hình và Bắt đầu Huấn luyện (Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import sys

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to="none",
        eval_strategy="epoch",
        save_strategy="epoch",
    ),
)

# Xử lý sửa lỗi TRL khi dùng processing_class
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config'] = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

# Chạy training
trainer_stats = trainer.train()



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,009 | Num Epochs = 3 | Total steps = 2,631
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.542722,1.462147
2,1.366787,1.424046
3,1.294456,1.428032


/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in 

In [6]:
# 5. Kiểm tra Inference với đoạn hội thoại
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Bạn là một chuyên gia y tế AI thông minh, tận tâm và có khả năng giải đáp các vấn đề sức khỏe một cách chính xác."},
    {"role": "user", "content": train_dataset[0]["question"]},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)

print("\n--- KẾT QUẢ MẪU TỪ MODEL ---\n")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_ma


--- KẾT QUẢ MẪU TỪ MODEL ---

system
Bạn là một chuyên gia y tế AI thông minh, tận tâm và có khả năng giải đáp các vấn đề sức khỏe một cách chính xác.
user
Đang chích ngừa viêm gan B có chích ngừa Covid-19 được không?
assistant
Nếu chị đang điều trị viêm gan B mạn tính, không phải dùng thuốc ức chế miễn dịch hoặc corticoid thì vẫn có thể tiêm phòng vaccine Covid-19 được. Tuy nhiên nếu chị đang sử dụng thuốc kháng retrovirus cần hoãn tiêm trong vòng 3 tháng sau khi kết thúc điều trị. Nếu chị đang dùng thuốc ức chế miễn dịch (tăng nguy cơ phản ứng phản vệ) hoặc corticoid (có thể ảnh hưởng đến chức năng tuyến giáp) thì nên hoãn tiêm phòng vaccine Covid-19.


In [7]:
# 6. Lưu Model
model.save_pretrained("qwen2.5-1.5b-instruct-lora-vihealthqa")
tokenizer.save_pretrained("qwen2.5-1.5b-instruct-lora-vihealthqa")
print("Đã lưu thành công model tại folder qwen2.5-1.5b-instruct-lora-vihealthqa!")

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-1.5b-instruct-lora-vihealthqa/tokenizer_config.json.


Đã lưu thành công model tại folder qwen2.5-1.5b-instruct-lora-vihealthqa!


# 7. Đánh giá mô hình (Evaluation) trên tập test.csv
Đoạn code sau đây sẽ load tập `test.csv`, chạy quá trình sinh từ (inference) để dự đoán các câu trả lời dựa vào tập câu hỏi. Sau đó dùng thư viện độ đo ROUGE (thường được dùng để đánh giá tóm tắt và sinh văn bản tự nhiên) để tự động tính điểm.

In [ ]:
!pip install rouge_score sacrebleu

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

# Tải dữ liệu test
test_df = pd.read_csv('test.csv')

# Lấy ngẫu nhiên 50 câu hỏi để test nhanh (do quá trình sinh câu trả lời tốn khá nhiều thời gian trên GPU)
# Nếu muốn test toàn bộ 2000+ câu, bạn có thể xóa đoạn `.sample(50, ...)` ở dưới
test_sample = test_df.sample(50, random_state=42).reset_index(drop=True)

print("Đang sinh câu trả lời cho tập test...")
FastLanguageModel.for_inference(model)

predictions = []
references = []

for i, (idx, row) in enumerate(test_sample.iterrows(), 1):
    print(f"Đang xử lý câu hỏi thứ {i}/{len(test_sample)}...")
    question = row['question']
    actual_answer = str(row['answer'])
    
    messages = [
        {"role": "system", "content": "Bạn là một chuyên gia y tế AI thông minh, tận tâm và có khả năng giải đáp các vấn đề sức khỏe một cách chính xác."},
        {"role": "user", "content": question},
    ]
    
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    # Generate text: Giới hạn độ dài câu trả lời tương đối 256-512 token để tiết kiệm thời gian
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True, pad_token_id=tokenizer.pad_token_id)
    input_length = inputs.shape[1]
    
    # Lọc bỏ special tokens để lấy text thô
    predicted_answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    
    predictions.append(predicted_answer)
    references.append(actual_answer)

# Lưu kết quả dự đoán ra file CSV để dễ dàng xem lại bằng mắt thường (Human Evaluation)
results_df = pd.DataFrame({
    'question': test_sample['question'],
    'actual_answer': references,
    'predicted_answer': predictions
})
results_df.to_csv("test_predictions.csv", index=False)


In [ ]:
from rouge_score import rouge_scorer

# Khởi tạo bộ tính điểm ROUGE (không dùng stemmer vì đây là tiếng Việt)
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

rouge1_f1 = 0
rouge2_f1 = 0
rougeL_f1 = 0

for pred, ref in zip(predictions, references):
    scores = scorer.score(ref, pred)
    rouge1_f1 += scores['rouge1'].fmeasure
    rouge2_f1 += scores['rouge2'].fmeasure
    rougeL_f1 += scores['rougeL'].fmeasure

n = len(predictions)
print("\n=== KẾT QUẢ ĐÁNH GIÁ TỰ ĐỘNG TRÊN TẬP TEST (ROUGE SCORE) ===")
print(f"ROUGE-1 F1: {rouge1_f1 / n:.4f}")
print(f"ROUGE-2 F1: {rouge2_f1 / n:.4f}")
print(f"ROUGE-L F1: {rougeL_f1 / n:.4f}")
print("-----------------------------------------------------------")
print("Đã lưu chi tiết so sánh các câu trả lời vào file 'test_predictions.csv'")